# Computergrafik Praxis - Groundtruth Dokumentation

Dieses Notebook ist sukzessiv nach Teilaufgaben aufgebaut. Jede Teilaufgabe ist einklappbar und enthaelt direkt am Ende ein eigenes Glossar mit den wichtigsten Begriffen und Abkuerzungen.

Stand: 2026-05-01

Quellen im Repository:

- `tasks/task02.md`
- `tasks/task03.md`
- `tasks/task03_dokumentation.md`
- `tasks/task04.md`
- `tasks_src/task02/task02.cpp`
- `tasks_src/task03/task03.cpp`
- `tasks_src/task04/task04.cpp`
- `assets/shaders/*.vert` und `assets/shaders/*.frag`

<details open>
<summary><strong>Aufgabe 2.1 - Vertex-Daten auf die GPU laden und rendern</strong></summary>

### Ziel

In Aufgabe 2 soll das Modell `stormtrooper.obj` geladen, in einen GPU-Buffer kopiert und mit OpenGL gerendert werden. Das vorhandene Geruest in `tasks_src/task02/task02.cpp` laedt bereits Fenster, Shader, Kamera und Modell. Die fehlenden Stellen sind VAO, VBO, Drawcall und Cleanup.

### Ausgangsdaten auf der CPU

Das Modell wird ueber `Model::Load("models/stormtrooper.obj")` geladen. Intern benutzt `Model::Load` den `tinyobjloader`. Aus den OBJ-Indizes entsteht eine flache Vertexliste:

```cpp
const std::vector<Vertex>& vertices = model.GetVertices();
```

Der gemeinsame Vertex-Typ des Projekts ist:

```cpp
struct Vertex
{
    Vec3f position;
    Vec3f normal;
    Vec3f color;
    Vec3f uv;
};
```

Fuer Aufgabe 2 liest der Shader nur `position`. Trotzdem wird der ganze `Vertex`-Struct in den VBO kopiert. Nicht gelesen werden `normal`, `color` und `uv`, weil `task02.vert` nur dieses Attribut besitzt:

```glsl
layout (location = 0) in vec3 in_position;
```

### VBO, VAO und Shader-Zusammenhang

```mermaid
flowchart LR
    OBJ[stormtrooper.obj] --> LOAD[Model::Load]
    LOAD --> CPU[CPU vector<Vertex>]
    CPU --> VBO[VBO: rohe Vertexbytes auf GPU]
    VBO --> VAO[VAO: liest position ab Offset 0]
    VAO --> VS[Vertexshader location 0: in_position]
    VS --> DRAW[glDrawArrays GL_TRIANGLES]
```

Merksatz:

```text
VBO = Speicher mit Vertexdaten
VAO = Anleitung, wie dieser Speicher gelesen wird
Shader = Empfaenger der gelesenen Attribute
```

### Code fuer den GPU-Upload

```cpp
GLuint vbo = 0;
glCreateBuffers(1, &vbo);

glNamedBufferData(
    vbo,
    model.GetVertices().size() * sizeof(Vertex),
    model.GetVertices().data(),
    GL_STATIC_DRAW);
```

### Code fuer das VAO

```cpp
GLuint vao = 0;
glCreateVertexArrays(1, &vao);

glVertexArrayVertexBuffer(vao, 0, vbo, 0, sizeof(Vertex));
glVertexArrayAttribFormat(vao, 0, 3, GL_FLOAT, GL_FALSE, 0);
glVertexArrayAttribBinding(vao, 0, 0);
glEnableVertexArrayAttrib(vao, 0);
```

Die Position steht im `Vertex` direkt am Anfang. Deshalb ist der relative Offset `0`. Der Abstand zum naechsten Vertex ist `sizeof(Vertex)`.

### Rendern

Im Renderloop werden Model-, View- und Projection-Matrix gesetzt. Danach wird gezeichnet:

```cpp
shader.Use();
glBindVertexArray(vao);

glUniformMatrix4fv(0, 1, GL_FALSE, modelMat.Data());
glUniformMatrix4fv(1, 1, GL_FALSE, viewMat.Data());
glUniformMatrix4fv(2, 1, GL_FALSE, projMat.Data());

glDrawArrays(GL_TRIANGLES, 0, (GLsizei)model.NumVertices());
```

`glDrawArrays` passt hier, weil `Model::Load` eine expandierte Dreiecksliste erzeugt. Es wird kein Indexbuffer gebraucht.

### Cleanup

```cpp
glDeleteBuffers(1, &vbo);
glDeleteVertexArrays(1, &vao);
```

### Glossar

- **CPU**: Hauptprozessor. Laedt das Modell und schickt Daten zur GPU.
- **GPU**: Grafikprozessor. Fuehrt Shader aus und rendert das Bild.
- **Vertex**: Ein Punkt eines Meshes mit Zusatzdaten wie Position, Normale oder UV.
- **VBO**: Vertex Buffer Object. Speicher fuer Vertexdaten auf der GPU.
- **VAO**: Vertex Array Object. Beschreibt, wie Daten aus dem VBO als Shaderattribute gelesen werden.
- **Offset**: Byteposition eines Feldes innerhalb eines Vertex.
- **Stride**: Byteabstand von einem Vertex zum naechsten.
- **Shader**: GPU-Programm. Aufgabe 2 nutzt Vertex- und Fragmentshader.
- **Uniform**: Shaderwert, der fuer den ganzen Drawcall gleich bleibt, zum Beispiel eine Matrix.
- **Drawcall**: Zeichenbefehl an OpenGL, zum Beispiel `glDrawArrays`.
- **OBJ**: 3D-Modellformat.
- **tinyobjloader**: Bibliothek zum Lesen von OBJ-Dateien.

</details>

<details>
<summary><strong>Aufgabe 2.2 - Dokumentation der OpenGL-Funktionen</strong></summary>

### Ziel

Aufgabe 2.2 verlangt eine kurze, ernsthafte Dokumentation der verwendeten OpenGL-Funktionen. Wichtig ist nicht nur, was jede Funktion einzeln tut, sondern wie sie zusammenarbeiten.

### Zusammenhang der Funktionen

```mermaid
flowchart TB
    CBUF[glCreateBuffers] --> DATA[glNamedBufferData / glNamedBufferSubData]
    CVAO[glCreateVertexArrays] --> BIND[glVertexArrayVertexBuffer]
    DATA --> BIND
    BIND --> FORMAT[glVertexArrayAttribFormat]
    FORMAT --> ABIND[glVertexArrayAttribBinding]
    ABIND --> ENABLE[glEnableVertexArrayAttrib]
    ENABLE --> USE[glBindVertexArray]
    USE --> DRAW[glDrawArrays]
```

### Funktionsbedeutung

- `glCreateBuffers`: Erzeugt echte, initialisierte Buffer-Objekte.
- `glNamedBufferData`: Reserviert Speicher und kann Initialdaten direkt kopieren.
- `glNamedBufferSubData`: Kopiert Daten in einen bereits angelegten Bufferbereich.
- `glCreateVertexArrays`: Erzeugt ein VAO.
- `glVertexArrayVertexBuffer`: Verbindet einen VBO mit einem VAO-Bindingpunkt.
- `glVertexArrayAttribFormat`: Beschreibt das Format eines Attributs.
- `glVertexArrayAttribBinding`: Verbindet Attributlocation und Bindingpunkt.
- `glEnableVertexArrayAttrib`: Aktiviert ein Attribut.
- `glBindVertexArray`: Macht das VAO fuer folgende Drawcalls aktiv.
- `glDrawArrays`: Zeichnet eine nicht-indizierte Vertexliste.

### `glCreateBuffers` vs. `glGenBuffers`

`glCreateBuffers` gehoert zum modernen Direct-State-Access-Stil. Es erzeugt direkt ein initialisiertes Buffer-Objekt. `glGenBuffers` stammt aus aelterem OpenGL und reserviert historisch zunaechst nur Namen; das Objekt entsteht durch spaeteres Binden an ein Target. Die Aufgabenstellung fordert deshalb `glCreateBuffers`.

### Glossar

- **API**: Programmierschnittstelle, hier OpenGL.
- **OpenGL State**: Zustand, den OpenGL intern speichert, zum Beispiel aktives VAO.
- **DSA**: Direct State Access. Objekte werden direkt ueber ihre IDs veraendert.
- **Binding**: Verknuepfung eines Objekts mit einem Slot oder Zustand.
- **RenderDoc**: Grafikdebugger zum Untersuchen von Frames, Buffern und Drawcalls.
- **Resource Cleanup**: Freigeben von GPU-Objekten mit `glDelete...`.

</details>

<details>
<summary><strong>Aufgabe 3.0 - Setup</strong></summary>

### Ziel

Das aktuelle Rahmenprogramm wird frisch geklont, weil Aufgabe 3 auf einer aktualisierten Basis aufsetzt. Danach wird `tasks_src/task03/task03.cpp` gebaut und mit dem Asset-Pfad gestartet.

### Wichtige Schritte

```powershell
cmake -B build -S .
cmake --build build --target task03 --config Debug
.\build\Debug\task03.exe assets\
```

Das Programm soll nach dem Start ein Koordinatensystem bzw. spaeter die Aufgabe-3-Szene anzeigen.

### Glossar

- **Clone**: Lokale Kopie eines Git-Repositories.
- **CMake**: Werkzeug, das Build-Dateien erzeugt.
- **Target**: Konkretes Bauziel, zum Beispiel `task03`.
- **Assets**: Laufzeitdaten wie Shader, Modelle und Texturen.
- **Working Directory**: Arbeitsordner, von dem relative Pfade ausgehen.

</details>

<details>
<summary><strong>Aufgabe 3.1 - Farbattribute</strong></summary>

### Ziel

Die Vertices enthalten nicht nur Position und Normale, sondern auch eine Farbe. Diese Farbe soll vom VBO ueber das VAO in den Vertexshader und von dort in den Fragmentshader gelangen.

### Datenweg

```mermaid
flowchart LR
    V[Vertex.color] --> VBO[VBO]
    VBO --> VAO[VAO Location 2\nOffset 6 floats]
    VAO --> VS[Vertexshader in_Color]
    VS --> OUT[out_Color]
    OUT --> FS[Fragmentshader in_Color]
    FS --> PIXEL[Pixel-Farbe]
```

### Layout

- Location 0: Position
- Location 1: Normalenvektor
- Location 2: Farbe

`color` beginnt nach `position` und `normal`, also nach `6 * sizeof(float)`.

### Glossar

- **Farbattribut**: Farbe, die pro Vertex gespeichert ist.
- **RGB**: Rot, Gruen, Blau.
- **Location**: Fester Slot eines Shaderattributes.
- **Interpolation**: OpenGL mischt Werte ueber ein Dreieck fuer jedes Fragment.
- **Fragment**: Kandidat fuer einen finalen Pixel.

</details>

<details>
<summary><strong>Aufgabe 3.2 - Primitive: Quader, Zylinder, Kugel</strong></summary>

### Ziel

Es werden Funktionen gebaut, die Quader, Zylinder und Kugel als Dreieckslisten erzeugen. Alle Primitive liegen zuerst lokal um den Ursprung. Groesse und Position entstehen spaeter ueber Matrizen.

### Gemeinsames Prinzip

```mermaid
flowchart TB
    P[Primitive erzeugen] --> POS[position lokal]
    P --> N[normal lokal]
    P --> C[color]
    POS --> V[vector<Vertex>]
    N --> V
    C --> V
    V --> GPU[VBO auf GPU]
```

### Quader

Der Quader nutzt acht Eckpunkte mit halber Ausdehnung `0.5`. Jede Flaeche besteht aus zwei Dreiecken. Insgesamt entstehen `6 * 2 * 3 = 36` Vertices. Jede Flaeche bekommt eigene Vertices, damit jede Flaeche eine eigene konstante Normale haben kann. Dadurch bleiben die Kanten hart.

### Zylinder

Der Zylinder liegt entlang der Z-Achse. Die Kreisringe werden ueber Winkel erzeugt:

```cpp
angle0 = twoPi * i / segments;
angle1 = twoPi * (i + 1) / segments;
ring = (cos(angle) * radius, sin(angle) * radius, 0);
```

Der Mantel bekommt radiale Normalen, Deckel und Boden bekommen konstante Normalen.

### Kugel

Die Kugel ist eine UV-Sphere aus Stacks und Slices:

```mermaid
flowchart TB
    START[slices + stacks] --> STACK[phi von 0 bis pi]
    STACK --> SLICE[theta von 0 bis 2pi]
    SLICE --> PTS[p00 p10 p01 p11]
    PTS --> TRIS[Dreiecke]
    TRIS --> SPHERE[Kugel-Mesh]
```

Kugelkoordinaten:

```cpp
x = radius * cos(theta) * sin(phi);
y = radius * sin(theta) * sin(phi);
z = radius * cos(phi);
```

Die Normale ist `Normalize(position)`, weil die Kugel im Ursprung liegt.

### Winding

Da `glFrontFace(GL_CCW)` und `GL_CULL_FACE` aktiv sind, muss die Dreiecksreihenfolge von aussen betrachtet gegen den Uhrzeigersinn sein. Sonst werden sichtbare Flaechen als Rueckseiten behandelt.

### Glossar

- **Primitive**: Einfache Grundkoerper.
- **Dreiecksliste**: Vertexliste, in der je drei Vertices ein Dreieck bilden.
- **Normale**: Richtung senkrecht zur Oberflaeche.
- **Stack**: Kugel-Unterteilung von Pol zu Pol.
- **Slice**: Kugel-Unterteilung rundherum.
- **Theta**: Rundum-Winkel von 0 bis 2 pi.
- **Phi**: Polarwinkel von 0 bis pi.
- **Winding**: Reihenfolge der Dreieckspunkte.
- **CCW**: Counter-clockwise, gegen den Uhrzeigersinn.
- **Face Culling**: Entfernen von Rueckseiten.

</details>

<details>
<summary><strong>Aufgabe 3.3 - Rendern der Geometrie</strong></summary>

### Ziel

Die erzeugten Primitive werden auf die GPU geladen und in der Szene gerendert. Quader, Zylinder, Kugel und Lichtmarker besitzen jeweils eigene VBOs, aber dasselbe VAO-Layout.

### Mehrere VBOs, ein VAO

```mermaid
flowchart LR
    C[cuboid VBO] --> VAO[gleiches VAO-Layout]
    Y[cylinder VBO] --> VAO
    S[sphere VBO] --> VAO
    VAO --> SHADER[Shader Locations 0,1,2]
    SHADER --> DRAW[glDrawArrays]
```

Vor jedem Drawcall wird Binding 0 auf den passenden VBO gesetzt:

```cpp
glVertexArrayVertexBuffer(VAO, 0, meshes[meshIndex].vbo, 0, sizeof(Vertex));
```

Das Format bleibt gleich:

- Location 0: Position
- Location 1: Normale
- Location 2: Farbe

### Glossar

- **VBO**: Speicher fuer Vertexdaten.
- **VAO**: Layoutbeschreibung fuer Vertexattribute.
- **Bindingpunkt**: Slot im VAO, an den ein VBO gebunden wird.
- **Drawcall**: Zeichenbefehl.
- **Model-Matrix**: Positioniert, skaliert oder rotiert ein Objekt.

</details>

<details>
<summary><strong>Aufgabe 3.4 - Debugging der Normalenvektoren</strong></summary>

### Ziel

Es soll geprueft werden, ob Normalen richtig erzeugt und richtig transformiert werden.

### Methode 1: Normalen als Farben

Normalen liegen im Bereich `[-1, 1]`, Farben im Bereich `[0, 1]`. Deshalb wird umgerechnet:

```glsl
vec3 normalColor = normalize(in_Normal) * 0.5 + 0.5;
outColor = vec4(normalColor, 1.0);
```

### Methode 2: Normalenlinien

Aus Stichproben der Vertices werden Linien erzeugt:

```cpp
endPosition = position + Normalize(normal) * length;
```

```mermaid
flowchart LR
    V[Vertex position] --> START[Linienstart]
    N[Vertex normal] --> END[position + normal * length]
    START --> LINE[GL_LINES]
    END --> LINE
```

### Glossar

- **Normalen-Debugging**: Sichtbarmachen von Normalen.
- **Normalenlinie**: Linie in Normalenrichtung.
- **GL_LINES**: OpenGL zeichnet je zwei Vertices als Linie.
- **Remapping**: Umrechnung von `[-1,1]` nach `[0,1]`.
- **ImGui**: UI-Bibliothek fuer Debugfenster.

</details>

<details>
<summary><strong>Aufgabe 3.5 - Beleuchtung</strong></summary>

### Ziel

Die Szene wird mit einer Punktlichtquelle und Lambert-Beleuchtung gerendert.

### Lambert-Modell

```glsl
vec3 lightDir = normalize(u_LightViewPos - in_ViewSpacePos);
float diffuse = max(dot(normal, lightDir), 0.0);
vec3 ambient = 0.18 * in_Color;
vec3 litColor = ambient + diffuse * in_Color;
```

### Koordinatenraum

Fragmentposition, Normale und Lichtposition werden im View-Space benutzt. Das ist wichtig, weil Beleuchtungsrechnung nur korrekt ist, wenn alle beteiligten Vektoren im selben Raum liegen.

### Normal-Matrix

Bei nicht-uniformer Skalierung duerfen Normalen nicht einfach mit der Model-View-Matrix transformiert werden. Der Vertexshader nutzt:

```glsl
mat3 normalMat = transpose(inverse(mat3(modelViewMat)));
out_Normal = normalize(normalMat * in_Normal);
```

### Glossar

- **Lambert**: Diffuses Beleuchtungsmodell.
- **Punktlichtquelle**: Licht mit Position im Raum.
- **Dot Product**: Skalarprodukt; misst Winkelnaehe zweier Richtungen.
- **Ambient**: Konstanter Grundlichtanteil.
- **View-Space**: Koordinatensystem aus Kamerasicht.
- **Normal-Matrix**: Inverse transponierte Matrix fuer Normalen.

</details>

<details>
<summary><strong>Aufgabe 3.6 - Bewegung der Kamera</strong></summary>

### Ziel

Die Kamera wird per Tastatur entlang ihrer eigenen Achsen bewegt und rotiert. Die Bewegung ist framerate-unabhaengig.

### Steuerung

- `W/S`: vor/zurueck
- `A/D`: links/rechts
- `Q/E`: runter/hoch
- Pfeile: Yaw/Pitch
- `R/F`: Roll
- `Shift`: schneller

### Delta-Time

```cpp
moveStep = moveSpeed * deltaSeconds;
rotateStep = rotateSpeed * deltaSeconds;
```

Damit haengt die Geschwindigkeit von Sekunden ab, nicht von Frames.

### Glossar

- **Kameraachsen**: Forward, Right, Up.
- **Yaw**: Links/rechts drehen.
- **Pitch**: Hoch/runter drehen.
- **Roll**: Um die Blickrichtung drehen.
- **Delta-Time**: Zeit seit letztem Frame.
- **SDL Scancode**: Physische Taste in SDL.

</details>

<details>
<summary><strong>Aufgabe 3.7 - Matrizen-Stack und animierte Szene</strong></summary>

### Ziel

Eine animierte Kran-Szene wird hierarchisch mit einem MatrixStack aufgebaut.

### Hierarchie

```mermaid
flowchart TB
    ROOT[Kran-Wurzel] --> BASE[Basis]
    ROOT --> TOWER[Turm]
    ROOT --> ARMROOT[Ausleger rotiert]
    ARMROOT --> ARM[Ausleger]
    ARMROOT --> HOOK[Hakenpunkt]
    HOOK --> ROPE[Seil]
    HOOK --> LOAD[Last]
```

### MatrixStack

- `Push`: aktuelle Matrix kopieren
- `Pop`: zur Elternmatrix zurueck
- `Last`: aktuelle Matrix
- `Translate`, `Rotate`, `Scale`: Transformationen anh?ngen

### Animation

```cpp
craneYaw = sinf(elapsedSeconds * 0.7f) * 35.0f;
```

Dadurch schwenkt der Ausleger periodisch.

### Glossar

- **MatrixStack**: Stapel von Transformationsmatrizen.
- **Hierarchie**: Kindobjekte erben Elterntransformationen.
- **Translate**: Verschieben.
- **Rotate**: Drehen.
- **Scale**: Skalieren.
- **Animation**: Zeitabhaengige Veraenderung.
- **Stack-Tiefe**: Anzahl verschachtelter Matrizen.

</details>

<details>
<summary><strong>Aufgabe 3.8 - Ship It</strong></summary>

### Ziel

Mit `shipit.bat` wird ein Release-Build erzeugt und zusammen mit DLLs und Assets gepackt.

### Befehl

```powershell
shipit.bat tasks_src\task03\
```

Das Ergebnis liegt unter:

```text
release_build\task03-Win-x64.zip
```

### Inhalt

- `task03.exe`
- `SDL3.dll`
- `physfs.dll`
- `assets/`

### Glossar

- **Release-Build**: Optimierte Buildvariante fuer Abgabe.
- **ZIP**: Gepacktes Archiv.
- **DLL**: Dynamische Windows-Bibliothek.
- **EXE**: Ausfuehrbare Windows-Datei.
- **Target**: Konkretes CMake-Bauziel.

</details>

<details>
<summary><strong>Aufgabe 4.0 - Setup Texturen</strong></summary>

### Ziel

Aufgabe 4 startet mit `tasks_src/task04/task04.cpp`. Das Programm laedt Shader, ein Bild und rendert zuerst ein rotes Quad.

### Build und Run

```powershell
cmake -B build -S .
cmake --build build --target task04 --config Debug
.\build\Debug\task04.exe assets\
```

### Glossar

- **Textur**: Bilddaten, die auf Geometrie gelegt werden.
- **Quad**: Viereck aus zwei Dreiecken.
- **Asset-Pfad**: Pfad zu Shadern, Texturen und Modellen.
- **SDL**: Bibliothek fuer Fenster und Eingaben.

</details>

<details>
<summary><strong>Aufgabe 4.1 - Texturkoordinaten fuer ein Quad</strong></summary>

### Ziel

Die vier Quad-Vertices bekommen UV-Koordinaten. Position beschreibt den Ort im Raum, UV beschreibt den Ort auf dem Bild.

### Aktueller Code

```cpp
links unten  -> uv = (0, 1)
rechts unten -> uv = (1, 1)
rechts oben  -> uv = (1, 0)
links oben   -> uv = (0, 0)
```

Die `v`-Werte sind gedreht, damit das Bild nicht auf dem Kopf steht.

### Diagramm

```mermaid
flowchart LR
    Q0[Quad links unten] --> U0[UV 0,1]
    Q1[Quad rechts unten] --> U1[UV 1,1]
    Q2[Quad rechts oben] --> U2[UV 1,0]
    Q3[Quad links oben] --> U3[UV 0,0]
```

### Glossar

- **UV**: Texturkoordinate. `u` horizontal, `v` vertikal.
- **Texel**: Ein Bildpunkt einer Textur.
- **Mapping**: Zuordnung von Geometrie zu Bildpositionen.
- **Quad**: Viereck aus zwei Dreiecken.
- **Bildursprung**: Stelle, an der Bildkoordinaten beginnen, oft oben links.

</details>

<details>
<summary><strong>Aufgabe 4.2 - OpenGL-Textur auf der GPU erstellen</strong></summary>

### Ziel

Das geladene Bild wird als OpenGL-Textur auf die GPU kopiert.

### Datenfluss

```mermaid
flowchart LR
    JPG[world map jpg] --> IMG[Image::Load]
    IMG --> CPU[CPU RGBA Pixel]
    CPU --> TEX[OpenGL Texture Object]
    TEX --> GPU[GPU Speicher]
```

### Relevanter Code

```cpp
GLuint textureHandle = 0;
glCreateTextures(GL_TEXTURE_2D, 1, &textureHandle);
glTextureParameteri(textureHandle, GL_TEXTURE_WRAP_S, GL_CLAMP_TO_EDGE);
glTextureParameteri(textureHandle, GL_TEXTURE_WRAP_T, GL_CLAMP_TO_EDGE);
glTextureParameteri(textureHandle, GL_TEXTURE_MIN_FILTER, GL_NEAREST);
glTextureParameteri(textureHandle, GL_TEXTURE_MAG_FILTER, GL_NEAREST);
glTextureStorage2D(textureHandle, 1, GL_RGBA8, image.GetWidth(), image.GetHeight());
glTextureSubImage2D(textureHandle, 0, 0, 0, image.GetWidth(), image.GetHeight(), GL_RGBA, GL_UNSIGNED_BYTE, image.Data());
```

### Glossar

- **Texture Object**: OpenGL-Objekt fuer Texturdaten.
- **RGBA8**: 8 Bit pro Rot-, Gruen-, Blau- und Alpha-Kanal.
- **Wrap Mode**: Verhalten ausserhalb UV `[0,1]`.
- **GL_CLAMP_TO_EDGE**: Randpixel werden weiterverwendet.
- **Filter**: Auswahl/Mischung von Texeln.
- **GL_NEAREST**: Naechster Texel wird genommen.
- **stb_image**: Bibliothek zum Laden von Bilddateien.

</details>

<details>
<summary><strong>Aufgabe 4.3 - Texturkoordinaten im Shader zugaenglich machen</strong></summary>

### Ziel

Die UV-Koordinaten aus dem VBO werden ueber das VAO an den Vertexshader und weiter an den Fragmentshader gegeben.

### Datenfluss

```mermaid
flowchart LR
    V[Vertex.uv im VBO] --> VAO[VAO Location 3\nOffset 9 floats]
    VAO --> VS[in_UV]
    VS --> OUT[out_UV]
    OUT --> R[Rasterizer interpoliert]
    R --> FS[in_UV]
```

### Shadercode

Vertexshader:

```glsl
layout(location = 3) in vec3 in_UV;
layout(location = 3) out vec3 out_UV;
out_UV = in_UV;
```

Fragmentshader:

```glsl
layout(location = 3) in vec3 in_UV;
```

### Glossar

- **Shaderattribute**: Eingaben des Vertexshaders.
- **Location 3**: Slot fuer UV-Koordinaten.
- **Interpolation**: Zwischenwerte ueber ein Dreieck.
- **Rasterizer**: Pipeline-Stufe, die Dreiecke in Fragmente zerlegt.
- **Swizzle**: Zugriff wie `.st` auf Vektorkomponenten.

</details>

<details>
<summary><strong>Aufgabe 4.4 - Textur im Fragmentshader samplen</strong></summary>

### Ziel

Der Fragmentshader liest mit den interpolierten UV-Koordinaten die passende Farbe aus der Textur.

### Code

```glsl
layout(binding = 0) uniform sampler2D u_Texture;

void main()
{
    vec4 texColor = texture(u_Texture, in_UV.st);
    outColor = texColor;
}
```

CPU-seitig wird die Textur gebunden:

```cpp
glBindTextureUnit(0, textureHandle);
```

### Diagramm

```mermaid
flowchart LR
    UV[in_UV.st] --> SAMPLE[texture]
    TEX[u_Texture Binding 0] --> SAMPLE
    SAMPLE --> COLOR[outColor]
```

### Glossar

- **Sampler**: Shader-Zugriff auf eine Textur.
- **sampler2D**: Sampler fuer 2D-Texturen.
- **Texture Unit**: Bindeslot fuer Texturen.
- **Binding 0**: Slot, an dem `u_Texture` die Textur erwartet.
- **Sampling**: Auslesen einer Texturfarbe an UV-Koordinaten.

</details>

<details>
<summary><strong>Aufgabe 4.5 - Texturkoordinaten fuer die Kugel</strong></summary>

### Ziel

Die Kugel aus Aufgabe 3 wird so erweitert, dass sie UV-Koordinaten besitzt. Dadurch kann eine Weltkarte auf die Kugel gelegt werden.

### Weltkarte

Die Datei liegt hier:

```text
assets/textures/world.200404.3x5400x2700.jpg
```

Sie hat ein 2:1-Seitenverhaeltnis. Das passt zu einer equirectangular world map: links/rechts ist einmal um die Erde herum, oben/unten ist Nordpol bis Suedpol.

### Kugelparametrisierung

Die Kugel wird ueber zwei Winkel erzeugt:

- `theta`: Rundum-Winkel von `0` bis `2*pi`.
- `phi`: Polarwinkel von `0` bis `pi`.

3D-Position:

```cpp
x = radius * cos(theta) * sin(phi);
y = radius * sin(theta) * sin(phi);
z = radius * cos(phi);
```

UV-Koordinaten:

```cpp
u = theta / (2*pi);
v = phi / pi;
```

### Aufbau aus Stacks und Slices

```mermaid
flowchart TB
    START[slices + stacks] --> STACK[phi-Schleife]
    STACK --> SLICE[theta-Schleife]
    SLICE --> POINTS[p00 p10 p01 p11]
    POINTS --> UVS[UV aus theta/phi]
    UVS --> TRIS[Dreiecke]
    TRIS --> VBO[Kugelvertices in VBO]
```

### Polfaelle

Am Nord- und Suedpol fallen mehrere Ringpunkte zusammen. Deshalb wird im ersten und letzten Stack jeweils nur ein Dreieck pro Slice erzeugt. In der Mitte werden zwei Dreiecke pro Gitterzelle erzeugt.

### Drawcall

Die Kugel wird als fertige Dreiecksliste erzeugt, deshalb wird kein EBO gebraucht:

```cpp
glDrawArrays(GL_TRIANGLES, 0, (GLsizei)sphereVertices.size());
```

### Typische Fehler

- Kugel unsichtbar: Winding passt nicht zu `glFrontFace(GL_CCW)`.
- Textur kopfueber: `v` als `1.0f - phi / pi` testen.
- Naht sichtbar: normal bei `u = 0` / `u = 1`.
- Verzerrung an Polen: normal bei equirectangular Mapping.

### Glossar

- **UV-Sphere**: Kugel aus Laengen- und Breitengraden.
- **Equirectangular**: Rechteckige Weltkarte fuer Kugelmapping.
- **Theta**: Rundum-Winkel.
- **Phi**: Winkel von Nord nach Sued.
- **Stack**: horizontale Kugelunterteilung.
- **Slice**: vertikale/rundum Kugelunterteilung.
- **Naht**: Stelle, an der `u = 0` und `u = 1` zusammentreffen.
- **Polverzerrung**: Verzerrung nahe Nord- und Suedpol.
- **Winding**: Reihenfolge der Dreieckspunkte.

</details>

<details>
<summary><strong>Aufgabe 4.6 - Rotation der Kugel</strong></summary>

### Ziel

Die Kugel soll mit Pfeiltasten rotiert werden. Diese Aufgabe ist nach dem Rueckbau der experimentellen Blickwinkel-Rotation noch offen bzw. sollte wieder als normale, einfache Aufgabenloesung umgesetzt werden.

### Einfache Zielidee

Man speichert zwei Winkel:

```cpp
float sphereYaw = 0.0f;
float spherePitch = 0.0f;
```

Die Pfeiltasten veraendern diese Winkel mit Delta-Time. Danach wird die Model-Matrix gebaut:

```cpp
modelMat = Rotate(RAMEN_WORLD_UP, sphereYaw) * Rotate(RAMEN_WORLD_RIGHT, spherePitch);
```

### Diagramm

```mermaid
flowchart LR
    KEYS[Pfeiltasten] --> ANGLES[Yaw/Pitch Winkel]
    ANGLES --> MODEL[Model-Matrix]
    MODEL --> VS[Vertexshader]
    VS --> SCREEN[rotierende Kugel]
```

### Glossar

- **Yaw**: Links/rechts-Drehung.
- **Pitch**: Hoch/runter-Drehung.
- **Model-Matrix**: Objekttransformation.
- **Delta-Time**: Zeit seit dem letzten Frame.
- **Framerate-unabhaengig**: Gleich schnell trotz unterschiedlicher FPS.

</details>

<details open>
<summary><strong>Aufgabe 5.0 - Setup Cubemapping</strong></summary>

### Ziel

`task05.cpp` wird als eigenes Target gebaut und gestartet. Das Programm mountet den `assets/` Pfad, initialisiert OpenGL und zeigt ein ImGui-Fenster an. Damit steht das Grundgeruest fuer Cubemapping und spaeteres Environment Mapping.

### Build und Start

```powershell
cmake -S . -B build
cmake --build build --target task05 --config Debug
.\build\Debug\task05.exe assets\
```

### Glossar

- **Target**: Konkretes Buildziel, hier `task05`.
- **Assets**: Laufzeitdaten wie Shader, Modelle und Texturen.
- **ImGui**: UI-Bibliothek fuer Debug- und Einstellfenster.
- **OpenGL Context**: Aktiver Grafikzustand, in dem OpenGL-Befehle ausgefuehrt werden.

</details>

<details open>
<summary><strong>Aufgabe 5.1.0 - Innengewendeter Wuerfel</strong></summary>

### Ziel

Fuer die Cubemap wird ein Wuerfel erzeugt, dessen Ursprung im Mittelpunkt liegt und dessen Frontfaces nach innen zeigen. Die Kamera steht spaeter innerhalb dieses Wuerfels, deshalb muessen von innen die Vorderseiten sichtbar sein.

### Geometrie

Der Wuerfel wird als expandierte Dreiecksliste mit 36 Vertices erzeugt:

```text
6 Flaechen * 2 Dreiecke * 3 Vertices = 36 Vertices
```

Jede Flaeche bekommt einen konstanten Normalenvektor. Dadurch kann der erste Testshader die Normalen als Farben visualisieren.

### Warum zeigen die Frontfaces nach innen?

Mit `GL_CULL_FACE` und `glCullFace(GL_BACK)` werden Rueckseiten verworfen. Eine normale Box waere von aussen sichtbar. Eine Cubemap wird aber von innen betrachtet. Deshalb muss die Vertex-Reihenfolge so gewaehlt werden, dass OpenGL die Innenflaechen als Frontfaces erkennt.

### GPU-Upload

Die Vertexdaten werden in einen VBO kopiert und ueber ein VAO mit dem Shader verbunden:

- Location 0: Position
- Location 1: Normale

### Erster Shadertest

Zum Test wird noch keine Cubemap gesampelt. Stattdessen werden die Normalen im Fragmentshader in den Farbbereich `[0, 1]` umgerechnet:

```glsl
vec3 normalColor = normalize(in_Normal) * 0.5 + 0.5;
outColor = vec4(normalColor, 1.0);
```

### Richtungsachsen verstehen

Die drei Raumachsen beschreiben die drei Grundrichtungen des 3D-Raums:

- **X-Achse**: links und rechts
- **Y-Achse**: unten und oben
- **Z-Achse**: hinten und vorne

Das Vorzeichen legt fest, in welche Richtung entlang der Achse gezeigt wird:

- **+X**: positive X-Richtung
- **-X**: negative X-Richtung
- **+Y**: positive Y-Richtung
- **-Y**: negative Y-Richtung
- **+Z**: positive Z-Richtung
- **-Z**: negative Z-Richtung

Bei einer Cubemap bekommt jede dieser sechs Richtungen genau eine Wuerfelseite. Ein spaeterer Richtungsvektor im Shader waehlt dann automatisch die passende Seite aus.

### Glossar

- **Cubemap**: Umgebungstextur aus sechs Bildern, eines pro Wuerfelseite.
- **Frontface**: Vorderseite eines Dreiecks gemaess seiner Vertex-Reihenfolge.
- **Backface Culling**: Verwerfen von Rueckseiten beim Rendern.
- **VBO**: Vertex Buffer Object, GPU-Speicher fuer Vertexdaten.
- **VAO**: Vertex Array Object, beschreibt das Layout der Vertexattribute.
- **Normalenvektor**: Vektor senkrecht auf einer Flaeche.
- **X/Y/Z-Achse**: Die drei Grundachsen des 3D-Raums.

</details>

<details open>
<summary><strong>Aufgabe 5.1.1 - Erstellen der Cubemap-Textur</strong></summary>

### Ziel

Die sechs Bilder der Umgebung werden in eine OpenGL-Textur vom Typ `GL_TEXTURE_CUBE_MAP` geladen. Diese Textur repraesentiert die sechs Blickrichtungen einer Umgebung.

### Geladene Bilder

Es werden sechs Bilddateien geladen:

- `posx.jpg`
- `negx.jpg`
- `posy.jpg`
- `negy.jpg`
- `posz.jpg`
- `negz.jpg`

Diese Dateien repraesentieren die sechs festen Richtungen der Cubemap.

### Erzeugen des Texturobjekts

```cpp
GLuint cubemapHandle = 0;
glCreateTextures(GL_TEXTURE_CUBE_MAP, 1, &cubemapHandle);
```

Damit wird ein OpenGL-Texturobjekt erzeugt, das spaeter sechs Faces enthaelt.

### Texturparameter

```cpp
glTextureParameteri(cubemapHandle, GL_TEXTURE_WRAP_S, GL_CLAMP_TO_EDGE);
glTextureParameteri(cubemapHandle, GL_TEXTURE_WRAP_T, GL_CLAMP_TO_EDGE);
glTextureParameteri(cubemapHandle, GL_TEXTURE_WRAP_R, GL_CLAMP_TO_EDGE);
glTextureParameteri(cubemapHandle, GL_TEXTURE_MIN_FILTER, GL_LINEAR);
glTextureParameteri(cubemapHandle, GL_TEXTURE_MAG_FILTER, GL_LINEAR);
```

`S`, `T` und `R` sind die drei Texturachsen der Cubemap. `GL_CLAMP_TO_EDGE` verhindert Randartefakte an den Naehten zwischen zwei Faces.

### Speicher anlegen

```cpp
glTextureStorage2D(cubemapHandle, 1, GL_RGBA8, width, height);
```

Obwohl es sechs Bilder gibt, reicht ein einziger Aufruf. OpenGL weiss durch den Texturtyp bereits, dass eine Cubemap aus sechs 2D-Faces besteht.

### Hochladen der sechs Faces

Jedes Bild wird mit `glTextureSubImage3D` hochgeladen. Bei einer Cubemap bezeichnet `zoffset`, welches Face beschrieben wird:

- `0` -> `+X`
- `1` -> `-X`
- `2` -> `+Y`
- `3` -> `-Y`
- `4` -> `+Z`
- `5` -> `-Z`

Deshalb spielt die Reihenfolge der hochgeladenen Bilder eine Rolle. Wenn zwei Bilder vertauscht werden, ist die Umgebung spaeter raeumlich falsch orientiert.

### Bedeutung der Parameter von `glTextureSubImage3D`

```cpp
glTextureSubImage3D(texture,
                    level,
                    xoffset,
                    yoffset,
                    zoffset,
                    width,
                    height,
                    depth,
                    format,
                    type,
                    data);
```

- **texture**: Handle der Zieltextur.
- **level**: Mipmap-Level, hier `0`.
- **xoffset**: Startposition innerhalb des Faces in X-Richtung.
- **yoffset**: Startposition innerhalb des Faces in Y-Richtung.
- **zoffset**: Face-Index der Cubemap.
- **width**: Breite des hochzuladenden Bildes.
- **height**: Hoehe des hochzuladenden Bildes.
- **depth**: Anzahl der zu beschreibenden Layer, hier `1`.
- **format**: Format der Quelldaten, hier `GL_RGBA`.
- **type**: Datentyp der Quelldaten, hier `GL_UNSIGNED_BYTE`.
- **data**: Zeiger auf die CPU-Bilddaten.

### Aktueller Stand

Die sechs Bilder werden jetzt geladen, als `GL_TEXTURE_CUBE_MAP` auf der GPU angelegt und auf die sechs Cubemap-Faces kopiert. Das eigentliche Sampling der Cubemap im Shader folgt in Aufgabe 5.1.2.

### Glossar

- **Face**: Eine der sechs Seiten einer Cubemap.
- **+X/-X/+Y/-Y/+Z/-Z**: Die sechs festen Richtungen des Cubemap-Raums.
- **Texture Handle**: OpenGL-ID eines Texturobjekts.
- **Sampler**: Shader-Zugriff auf eine Textur.
- **Mipmap-Level**: Aufloesungsstufe einer Textur.
- **Texel**: Ein einzelner Bildpunkt einer Textur.
- **DSA**: Direct State Access, direkter Zugriff auf OpenGL-Objekte ohne klassisches Binden.

</details>

<details open>
<summary><strong>Aufgabe 5.1.2 - Samplen von der Cubemap-Textur</strong></summary>

### Ziel

Die bereits erzeugte Cubemap soll nun beim Rendern auch wirklich verwendet werden. Dazu wird sie auf CPU-Seite an eine Texture Unit gebunden und im Fragmentshader ueber einen `samplerCube` abgefragt.

### Binden der Cubemap

```cpp
glBindTextureUnit(0, cubemapHandle);
```

Damit liegt die Cubemap auf Texture Unit `0`.

### Sampler im Fragmentshader

```glsl
layout(binding = 0) uniform samplerCube u_Cubemap;
```

Das `binding = 0` sorgt dafuer, dass der Shader genau dieselbe Texture Unit verwendet, an die auf CPU-Seite die Cubemap gebunden wurde.

### Richtung statt UV-Koordinaten

Im Gegensatz zu einer 2D-Textur braucht eine Cubemap keine UV-Koordinaten. Stattdessen wird ein 3D-Richtungsvektor benoetigt. Im Vertexshader wird dazu die Weltposition berechnet und als Richtung weitergereicht:

```glsl
vec4 worldPos = u_ModelMat * vec4(in_Position, 1.0f);
out_Direction = worldPos.xyz;
```

Im Fragmentshader wird diese Richtung normalisiert und anschliessend zum Sampling verwendet:

```glsl
vec3 sampleDir = normalize(in_Direction);
outColor = texture(u_Cubemap, sampleDir);
```

OpenGL waehlt daraufhin automatisch das passende Face `+X/-X/+Y/-Y/+Z/-Z` und die passende Stelle innerhalb dieses Faces aus.

### Warum funktioniert das?

Der innenliegende Wuerfel umschliesst die Kamera. Jede Position auf diesem Wuerfel zeigt vom Mittelpunkt aus in eine bestimmte Raumrichtung. Genau diese Richtung wird als Lookup-Vektor fuer die Cubemap verwendet.

### Aktueller Stand

Die Cubemap wird jetzt nicht nur auf die GPU geladen, sondern auch sichtbar dargestellt. Der Wuerfel zeigt damit die Umgebung statt der bisherigen Normalenfarben.

### Glossar

- **samplerCube**: GLSL-Samplertyp fuer Cubemap-Texturen.
- **Texture Unit**: Bindungsplatz, an den eine Textur vor dem Draw-Call gelegt wird.
- **Binding**: Feste Zuordnung zwischen Shader-Sampler und Texture Unit.
- **Uniform**: Shader-Eingabe, die pro Draw-Call von der CPU gesetzt wird.
- **Lookup-Vektor**: Richtungsvektor, mit dem in einer Cubemap gesucht wird.
- **World Position**: Position eines Vertex nach Anwendung der Model-Matrix.

</details>

<details open>
<summary><strong>Aufgabe 5.1.3 - Kamera</strong></summary>

### Ziel

Die Szene soll mit der Kamera erkundet werden koennen. Dafuer wird pro Frame der Tastaturzustand abgefragt und in Bewegungs- und Rotationsbefehle fuer die vorhandene `Camera`-Klasse uebersetzt.

### Eingabemodell

Der aktuelle Tastaturzustand wird mit `SDL_GetKeyboardState` ausgelesen. Das ist wichtig, weil eine Kamera nicht nur auf einzelne Tastendruecke reagieren soll, sondern kontinuierlich waehrend eine Taste gehalten wird.

```cpp
const bool* keyboardState = SDL_GetKeyboardState(nullptr);
```

### Framerate-unabhaengige Bewegung

```cpp
float deltaSeconds = (float)(msPerFrame / 1000.0);
float moveSpeed = 3.0f * deltaSeconds;
float rotateSpeed = 90.0f * deltaSeconds;
```

Damit wird die Bewegung in Einheiten pro Sekunde statt in Einheiten pro Frame beschrieben. So bleibt das Verhalten bei unterschiedlichen FPS moeglichst stabil.

### Bewegung im Raum

- `W` / `S`: `MoveForward`
- `A` / `D`: `MoveRight`
- `Q` / `E`: `MoveUp`

Diese Methoden verwenden die lokalen Kamerarichtungen `Forward`, `Right` und `Up`. Dadurch bewegt sich die Kamera immer relativ zu ihrer aktuellen Orientierung.

### Rotation

- Pfeil links / rechts: `Yaw`
- Pfeil hoch / runter: `Pitch`

`Yaw` dreht die Kamera um ihre lokale Hochachse. `Pitch` dreht sie um ihre Seitenachse. Dadurch kann die Blickrichtung frei geaendert werden.

### Zusammenhang mit der View-Matrix

Nach jeder Aenderung von Position oder Orientierung wird aus dem Kamerazustand wieder eine neue View-Matrix aufgebaut:

```cpp
Mat4f viewMat = LookAt(
    camera.GetPosition(),
    camera.GetPosition() + camera.GetForward(),
    camera.GetUp());
```

Damit wirken sich die Bewegungen direkt auf den sichtbaren Ausschnitt der Cubemap aus.

### Aktueller Stand

Die Umgebung kann nun frei erkundet werden. Dadurch laesst sich unmittelbar pruefen, ob die Cubemap in verschiedenen Blickrichtungen korrekt gesampelt wird.

### Glossar

- **deltaSeconds**: Vergangene Zeit seit dem letzten Frame in Sekunden.
- **Framerate-unabhaengig**: Verhalten ist nicht direkt von der Anzahl der Frames pro Sekunde abhaengig.
- **Yaw**: Drehung um die lokale Hochachse der Kamera.
- **Pitch**: Drehung um die Seitenachse der Kamera.
- **View-Matrix**: Matrix, die die Szene aus Sicht der Kamera beschreibt.
- **LookAt**: Funktion zum Aufbau einer View-Matrix aus Position, Blickrichtung und Up-Vektor.

</details>